### importing libraries

In [0]:

import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

### data ingestion

In [0]:

#importing data in spark library
data_path = spark.read.table("workspace.default.retail_sales_dataset")


## converting it to pandas
df = data_path.toPandas()

#showing top 10 rows in a dataframe
df.head(10)

In [0]:

# CREATE THE VISUALS 
fig, ax = plt.subplots(1, 2, figsize=(16, 6))

# Left side:Boxplot to compare products 
sns.boxplot(data=df, x='Product Category', y='Total Amount', ax=ax[0], palette='magma')
ax[0].set_title('Product Comparison:boxplot')
ax[0].tick_params(axis='x', rotation=45)

# Right side: Histogram to compare products 
sns.histplot(data=df, x='Total Amount', hue='Product Category', element='step', ax=ax[1])
ax[1].set_title('Product Comparison: Histogram')

plt.tight_layout()
plt.show()

# Identify Highest Variability (using Standard Deviation) 
variability = df.groupby('Product Category')['Total Amount'].std().sort_values(ascending=False)
print(f"1. Highest Variability: {variability.index[0]} (Std Dev: {variability.values[0]:.2f})")

# Identify Most Outliers (using IQR Method)
def get_outlier_count(column):
    q1, q3 = column.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((column < (q1 - 1.5 * iqr)) | (column > (q3 + 1.5 * iqr))).sum()

outliers = df.groupby('Product Category')['Total Amount'].apply(get_outlier_count).sort_values(ascending=False)
print(f"2. Most Outliers: {outliers.index[0]} ({outliers.values[0]} outliers detected)")